<a href="https://colab.research.google.com/github/ompatel420/FreeCodeCamp-Machine-Learning-Project/blob/main/Book_Recommendation_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Book_Recommendation_Engine Project**

In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

import os
import zipfile
import requests
from io import BytesIO

In [19]:
# Official FreeCodeCamp dataset URL
url = "https://cdn.freecodecamp.org/project-data/books/book-crossings.zip"

# Download dataset
response = requests.get(
    url,
    params={"downloadformat": "zip"},
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=120
)

response.raise_for_status()

# Extract dataset
with zipfile.ZipFile(BytesIO(response.content)) as zip_ref:
    zip_ref.extractall("/content/book-crossings")

# Dataset filenames
books_filename = "/content/book-crossings/BX-Books.csv"
ratings_filename = "/content/book-crossings/BX-Book-Ratings.csv"

print("Dataset downloaded successfully!")
print("Books file exists:", os.path.exists(books_filename))
print("Ratings file exists:", os.path.exists(ratings_filename))

Dataset downloaded successfully!
Books file exists: True
Ratings file exists: True


In [20]:
# Load books dataset
df_books = pd.read_csv(
    books_filename,
    encoding="ISO-8859-1",
    sep=";",
    header=0,
    names=["isbn", "title", "author"],
    usecols=["isbn", "title", "author"],
    dtype={
        "isbn": "str",
        "title": "str",
        "author": "str"
    }
)

# Load ratings dataset
df_ratings = pd.read_csv(
    ratings_filename,
    encoding="ISO-8859-1",
    sep=";",
    header=0,
    names=["user", "isbn", "rating"],
    usecols=["user", "isbn", "rating"],
    dtype={
        "user": "int32",
        "isbn": "str",
        "rating": "float32"
    }
)

print("Books dataset:", df_books.shape)
print("Ratings dataset:", df_ratings.shape)

print("\nBooks preview:")
display(df_books.head())

print("\nRatings preview:")
display(df_ratings.head())

Books dataset: (271379, 3)
Ratings dataset: (1149780, 3)

Books preview:


,isbn,title,author
0,0195153448,Classical Mythology,Mark P. O. Morford
1,0002005018,Clara Callan,Richard Bruce Wright
2,0060973129,Decision in Normandy,Carlo D'Este
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata
4,0393045218,The Mummies of Urumchi,E. J. W. Barber



Ratings preview:


,user,isbn,rating
0,276725,034545104X,0.0
1,276726,0155061224,5.0
2,276727,0446520802,0.0
3,276729,052165615X,3.0
4,276729,0521795028,6.0


In [21]:
# Copy ratings dataset
df = df_ratings.copy()

# Count ratings by users and books
counts1 = df["user"].value_counts()
counts2 = df["isbn"].value_counts()

# Remove users with fewer than 200 ratings
df = df[
    ~df["user"].isin(
        counts1[counts1 < 200].index
    )
]

# Remove books with fewer than 100 ratings
df = df[
    ~df["isbn"].isin(
        counts2[counts2 < 100].index
    )
]

# Merge ratings with book information
df = pd.merge(
    left=df_books,
    right=df,
    on="isbn"
)

# Remove duplicate user ratings for the same book title
df.drop_duplicates(
    subset=["title", "user"],
    inplace=True
)

print("Dataset cleaned successfully!")
print("Total records:", len(df))
print("Unique books:", df["title"].nunique())
print("Unique users:", df["user"].nunique())

Dataset cleaned successfully!
Total records: 49136
Unique books: 673
Unique users: 888


In [23]:
# Create pivot table
book_ratings = df.pivot(
    index="title",
    columns="user",
    values="rating"
).fillna(0)

# Convert to sparse matrix
book_ratings_matrix = csr_matrix(
    book_ratings.values
)

print("Book-rating matrix created successfully!")

print("Matrix shape:", book_ratings.shape)
print("Sparse matrix shape:", book_ratings_matrix.shape)

Book-rating matrix created successfully!
Matrix shape: (673, 888)
Sparse matrix shape: (673, 888)


In [24]:
# Create KNN model
model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_jobs=-1
)

# Train model
model.fit(book_ratings_matrix)

print("KNN model trained successfully!")

KNN model trained successfully!


In [25]:
def get_recommends(book=""):

    # Check if book exists
    if book not in book_ratings.index:
        raise ValueError(
            f"Book '{book}' not found in dataset."
        )

    # Get the selected book's rating vector
    book_index = book_ratings.index.get_loc(book)

    book_vector = book_ratings_matrix[book_index]

    # Find 6 nearest neighbors (including the selected book)
    distances, indices = model.kneighbors(
        book_vector,
        n_neighbors=6
    )

    recommended_books = []

    # Skip the original book
    for i in range(1, len(distances.flatten())):

        recommended_books.append([
            book_ratings.index[indices.flatten()[i]],
            float(distances.flatten()[i])
        ])

    # Reverse order to match expected FreeCodeCamp output
    recommended_books.reverse()

    return [book, recommended_books]

In [26]:
result = get_recommends(
    "The Queen of the Damned (Vampire Chronicles (Paperback))"
)

print("Selected Book:")
print(result[0])

print("\nFive Recommended Books:")

for book, distance in result[1]:

    print("\nBook:", book)
    print("Distance:", distance)

Selected Book:
The Queen of the Damned (Vampire Chronicles (Paperback))

Five Recommended Books:

Book: Catch 22
Distance: 0.793983519077301

Book: The Witching Hour (Lives of the Mayfair Witches)
Distance: 0.7448656558990479

Book: Interview with the Vampire
Distance: 0.7345068454742432

Book: The Tale of the Body Thief (Vampire Chronicles (Paperback))
Distance: 0.5376338362693787

Book: The Vampire Lestat (Vampire Chronicles, Book II)
Distance: 0.5178412199020386


In [27]:
def test_book_recommendation():

    test_pass = True

    # Test recommendation function
    recommends = get_recommends(
        "Where the Heart Is (Oprah's Book Club (Paperback))"
    )

    # Check original book
    if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
        test_pass = False

    # Expected recommended books
    recommended_books = [
        "I'll Be Seeing You",
        "The Weight of Water",
        "The Surgeon",
        "I Know This Much Is True"
    ]

    recommended_books_dist = [
        0.8,
        0.77,
        0.77,
        0.77
    ]

    # Check first two recommendations
    for i in range(2):

        if recommends[1][i][0] not in recommended_books:
            test_pass = False

        if abs(
            recommends[1][i][1] - recommended_books_dist[i]
        ) >= 0.05:
            test_pass = False

    if test_pass:
        print("You passed the challenge! 🎉🎉🎉🎉🎉")
    else:
        print("You haven't passed yet. Keep trying!")


# Run official test
test_book_recommendation()

You passed the challenge! 🎉🎉🎉🎉🎉
